In [1]:
import rose
import pandas as pd

In [2]:
%load_ext autoreload
%autoreload 2

In [45]:
# results = []

# for approach in [ rose.NeuralNetwork() ]: # rose.all_approaches():
#     print(f"Trying {approach.title} approach...")
#     df = rose.evaluate_approach(approach, verbose=1)
#     results.append(df)

#     rose.plot_learning(df)
#     rose.plot_scatter(approach, N=20)

# results = pd.concat(results, ignore_index=True)
# summary = rose.summarize_results(results)
# rose.format_summary(summary)

In [48]:
from pathlib import Path
import shutil

import pandas as pd
import matplotlib.pyplot as plt
from jinja2 import Template, Environment, FileSystemLoader

SUMMARY_TEMPLATE = "templates/summary.html"


def capture_plot(plot_function, path: Path, *args, **kwargs) -> None:
    original_show = plt.show

    try:
        plt.close("all")
        plt.show = lambda *a, **k: None

        plot_function(*args, **kwargs)

        fig = plt.gcf()
        fig.savefig(path, dpi=100, bbox_inches="tight")
        plt.close(fig)

    finally:
        plt.show = original_show


def format_percent(x):
    return "-" if pd.isna(x) else f"{100 * x:.2f}%"


def format_number(x):
    return "-" if pd.isna(x) else f"{x:.2f}"


def format_int(x):
    return "-" if pd.isna(x) else f"{int(x)}"


def format_summary_table(summary: pd.DataFrame) -> str:
    formatted = summary.rename(columns={
        "approach": "Approach",
        "min_N_for_100pct_accuracy": "Min N for 100% Accuracy",
        "accuracy_at_N_20": "Accuracy at N=20",
        "rmse_at_N_20": "RMSE at N=20",
        "max_accuracy_achieved": "Max Accuracy Achieved",
    }).copy()

    formatted["Min N for 100% Accuracy"] = formatted["Min N for 100% Accuracy"].map(format_int)
    formatted["Accuracy at N=20"] = formatted["Accuracy at N=20"].map(format_percent)
    formatted["RMSE at N=20"] = formatted["RMSE at N=20"].map(format_number)
    formatted["Max Accuracy Achieved"] = formatted["Max Accuracy Achieved"].map(format_percent)

    return formatted.to_html(
        index=False,
        classes="table table-striped table-sm align-middle",
        border=0,
        escape=False,
    )


def build_approach_report(
    approaches,
    *,
    output_dir: str = "approaches",
    scatter_N: int = 20,
    verbose: int = 1,
) -> pd.DataFrame:
    output_path = Path(output_dir)

    if output_path.exists():
        shutil.rmtree(output_path)

    output_path.mkdir(parents=True)

    results = []
    approach_contexts = []

    for approach in approaches:
        if verbose >= 1:
            print(f"Trying {approach.title} approach...")

        approach_dir = output_path / approach.code
        approach_dir.mkdir()

        df = rose.evaluate_approach(approach, verbose=verbose)
        results.append(df)

        df.to_csv(approach_dir / "results.csv", index=False)
        df.to_parquet(approach_dir / "results.parquet", index=False)

        learning_path = approach_dir / "learning.png"
        scatter_path = approach_dir / "scatter.png"

        capture_plot(rose.plot_learning, learning_path, df)
        capture_plot(rose.plot_scatter, scatter_path, approach, N=scatter_N)

        approach_contexts.append({
            "code": approach.code,
            "title": approach.title,
            "learning_plot": f"{approach.code}/learning.png",
            "scatter_plot": f"{approach.code}/scatter.png",
        })

    results = pd.concat(results, ignore_index=True)
    summary = rose.summarize_results(results)

    summary.to_csv(output_path / "summary.csv", index=False)
    summary.to_parquet(output_path / "summary.parquet", index=False)

    summary_by_title = summary.set_index("approach")

    for context in approach_contexts:
        row = summary_by_title.loc[context["title"]]

        context["metrics"] = {
            "Min N for 100% Accuracy": format_int(row["min_N_for_100pct_accuracy"]),
            "Accuracy at N=20": format_percent(row["accuracy_at_N_20"]),
            "RMSE at N=20": format_number(row["rmse_at_N_20"]),
            "Max Accuracy Achieved": format_percent(row["max_accuracy_achieved"]),
        }
        
    env = Environment(loader=FileSystemLoader("."))
    template = env.get_template(SUMMARY_TEMPLATE)

    html = template.render(
        summary_table=format_summary_table(summary),
        approaches=approach_contexts,
    )

    (output_path / "summary.html").write_text(html, encoding="utf-8")

    return summary

In [50]:
build_approach_report(
    rose.all_approaches(),
    verbose=1
)

Trying Naïve Linear Regression approach...
............................
Trying LR With Categorical Features approach...
.............stopping early at N=40
Trying Bincount Pivot approach...
.......stopping early at N=7
Trying GAM approach...
.

C:\Users\oloon\miniconda3\envs\py312\Lib\site-packages\pygam\pygam.py:1149: RuntimeWarning: divide by zero encountered in scalar divide
  r2["explained_deviance"] = 1.0 - full_d.sum() / null_d.sum()


...................stopping early at N=200
Trying GB Tree approach...
.....................stopping early at N=300
Trying Neural Network approach...
............................
Trying DeepSet approach...
...........stopping early at N=20


,approach,min_N_for_100pct_accuracy,accuracy_at_N_20,rmse_at_N_20,max_accuracy_achieved
0,Naïve Linear Regression,NaN,0.0964,4.051089e+00,0.1486
1,LR With Categorical Features,30.0,0.4062,9.199714e-01,1.0000
2,Bincount Pivot,6.0,1.0000,3.706762e-15,1.0000
3,GAM,100.0,0.1555,3.345492e+00,1.0000
4,GB Tree,200.0,0.0943,4.082113e+00,1.0000
5,Neural Network,NaN,0.0988,4.306416e+00,0.9044
6,DeepSet,5.0,1.0000,1.592346e-01,1.0000
